# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [2]:
import pandas as pd
import numpy as np
import faiss
import ollama

client = ollama.Client()

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [3]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


In [4]:
print("Total documents:", len(documents))
print("Data type:", type(documents))
print("First document:", documents[0])
print("Empty documents:", sum(not doc.strip() for doc in documents))

Total documents: 59
Data type: <class 'list'>
First document: Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.
Empty documents: 0


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan `nomic-embed-text:latest`.

In [16]:
def get_embedding(text):
    result = client.embed(
        model='nomic-embed-text:latest',  # sesuaikan versi model embedding
        input=f"search_document: {text}"
    )
    return result.embeddings[0]

def get_query_embedding(query):
    result = client.embed(
        model='nomic-embed-text:latest',
        input=f"search_query: {query}"
    )
    return result.embeddings[0]

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 768)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [17]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [18]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_query_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.49251363)}
{'rank': 2, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.51427275)}
{'rank': 3, 'question': 'Bagaimana cara melacak paket yang sedang dikirim?', 'answer': "Nomor resi akan muncul di halaman 'Pesanan Saya' setelah barang dikirim, dan dapat digunakan untuk melacak posisi paket di website kurir terkait.", 'category': 'Pengiriman', 'distance': np.float32(0.57892585)}


In [20]:
# Mencoba menjalankan Tiga Query

queries = [
    "Bagaimana cara mengembalikan barang?",
    "Saya ingin membatalkan pesanan yang sudah dibuat.",
    "Apakah platform ini menyediakan layanan perbaikan laptop?"
]

for query in queries:
    print(f"\nQuery: {query}")

    results = semantic_search(query, top_k=3)

    for result in results:
        print(
            f"Rank: {result['rank']} | "
            f"Category: {result['category']} | "
            f"Distance: {result['distance']:.4f}"
        )
        print(f"FAQ: {result['question']}")


Query: Bagaimana cara mengembalikan barang?
Rank: 1 | Category: Pengembalian | Distance: 0.5014
FAQ: Bagaimana prosedur pengembalian barang yang tidak sesuai?
Rank: 2 | Category: Pengembalian | Distance: 0.5353
FAQ: Bagaimana jika penjual menolak pengajuan pengembalian saya?
Rank: 3 | Category: Customer Service | Distance: 0.5388
FAQ: Bagaimana cara melaporkan penjual yang menipu?

Query: Saya ingin membatalkan pesanan yang sudah dibuat.
Rank: 1 | Category: Pemesanan | Distance: 0.3681
FAQ: Bagaimana cara membatalkan pesanan yang sudah dibuat?
Rank: 2 | Category: Produk | Distance: 0.5163
FAQ: Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
Rank: 3 | Category: Pemesanan | Distance: 0.5394
FAQ: Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?

Query: Apakah platform ini menyediakan layanan perbaikan laptop?
Rank: 1 | Category: Akun | Distance: 0.4704
FAQ: Apakah data pribadi saya aman di platform ini?
Rank: 2 | Category: Pemesanan | Distance: 0.4847
FA

## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [ ]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. 
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM (Local Ollama model: gpt-oss:20b)

In [22]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.chat(
        model='gpt-oss:20b',  # sesuaikan versi model
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.3
        }
    )
    answer = response.message.content

    return answer, search_results

# Testing

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Ya, Anda dapat menggunakan fitur cicilan. Cicilan tersedia untuk transaksi yang menggunakan kartu kredit tertentu dan pada nominal minimal yang ditentukan oleh pihak bank.

Sumber FAQ yang digunakan:
- Apakah biaya ongkos kirim pengembalian ditanggung pembeli?
- Apakah saya bisa membayar dengan cicilan?
- Bagaimana jika penjual menolak pengajuan pengembalian saya?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [28]:
test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    
    # 1. Direct query: pertanyaan yang jawabannya tersedia di FAQ
    "Apakah saya bisa membayar dengan cicilan?",

    # 2. Direct query: pertanyaan yang jawabannya tersedia di FAQ
    "Bagaimana cara melacak paket yang sedang dikirim?",

    # 3. Paraphrase: menggunakan kalimat berbeda dari FAQ asli
    "Kalau saya ingin produk dengan model tertentu, apakah bisa request langsung ke seller?",

    # 4. Paraphrase: menggunakan kalimat berbeda dari FAQ asli
    "Seusai melakukan transfer ke rekening penjual, apakah saya perlu mengirim bukti pembayaran?",

    # 5. Out-of-Knowledge-Base: informasi tidak tersedia di FAQ
    "Siapa yang membakar hutan di Kalimantan, Papua, dan Ujung Kulon?"
]

for q in test_queries:
    print('Query:', q)

    answer, sources = generate_answer(q)

    print('Jawaban:', answer)

    print('Sumber FAQ yang digunakan:')
    for s in sources:
        print('-', s['question'])

    print('---')


for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Apakah saya bisa membayar dengan cicilan?
Jawaban: Ya, Anda dapat membayar dengan cicilan. Fitur cicilan tersedia untuk transaksi yang menggunakan kartu kredit tertentu, dengan nominal minimal yang ditentukan oleh pihak bank.
Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah saya bisa memesan tanpa membuat akun?
- Apakah biaya ongkos kirim pengembalian ditanggung pembeli?
---
Query: Bagaimana cara melacak paket yang sedang dikirim?
Jawaban: Nomor resi akan muncul di halaman **Pesanan Saya** setelah barang Anda dikirim. Anda dapat menggunakan nomor resi tersebut untuk melacak posisi paket di website kurir terkait.
Sumber FAQ yang digunakan:
- Bagaimana cara melacak paket yang sedang dikirim?
- Bagaimana cara melaporkan penjual yang menipu?
- Bagaimana cara melakukan pemesanan di platform ini?
---
Query: Kalau saya ingin produk dengan model tertentu, apakah bisa request langsung ke seller?
Jawaban: Ya, beberapa penjual memang menyediakan opsi request 

### Catatan Analisis Relevansi


Berdasarkan lima skenario pengujian, sistem Semantic Search & LLM berhasil menjalankan proses retrieval dan generation sesuai dengan tujuan assignment.

Pada empat query yang memiliki jawaban dalam knowledge base, sistem berhasil menemukan FAQ yang relevan dalam tiga hasil teratas (top 3). Tiga query berhasil menempatkan FAQ yang relevan pada ranking pertama, sedangkan satu query mengenai konfirmasi transfer bank menempatkan FAQ yang relevan pada ranking ketiga.

Pengujian menggunakan parafrase menunjukkan bahwa semantic search mampu menangkap kesamaan makna meskipun pengguna menggunakan struktur kalimat dan kosakata yang berbeda dari FAQ asli. Hal ini terlihat pada pengujian request custom produk dan konfirmasi transfer bank.

Pada tahap generation, `gpt-oss:20b` mampu menyusun jawaban yang natural dan relevan berdasarkan informasi yang diberikan melalui context. Berdasarkan pemeriksaan terhadap dataset, jawaban untuk pertanyaan konfirmasi transfer bank juga sesuai dengan informasi FAQ yang tersedia.

Untuk pengujian di luar knowledge base, LLM berhasil menyatakan bahwa informasi yang diminta belum tersedia dan tidak memberikan jawaban spekulatif. Namun, FAISS tetap mengembalikan dokumen terdekat meskipun tidak relevan, karena mekanisme pencarian nearest neighbors tidak secara otomatis menentukan apakah suatu dokumen benar-benar menjawab pertanyaan.

Secara keseluruhan, prototype telah berhasil mengimplementasikan alur end-to-end mulai dari knowledge base, embedding, penyimpanan FAISS, semantic retrieval, hingga LLM-powered response. Peningkatan selanjutnya dapat difokuskan pada optimasi ranking retrieval dan penerapan mekanisme similarity threshold untuk membantu mendeteksi query yang tidak memiliki informasi relevan dalam knowledge base.